<div align="center">
  <img src="https://raw.githubusercontent.com/WenjieDu/PyPOTS/main/docs/figs/pypots_logos/PyPOTS_logo_trans.png" width="600" alt="PyPOTS logo"/>
  <h3>PyPOTS: A Python Toolbox for Data Mining on Partially-Observed Time Series</h3>
  <p>
    <a href="https://pypots.com">Website</a> ·
    <a href="https://docs.pypots.com">Docs</a> ·
    <a href="https://github.com/WenjieDu/PyPOTS">GitHub</a> ·
    <a href="https://github.com/WenjieDu/BrewPOTS">Tutorials</a>
  </p>
</div>

---


# KDD'26 Tutorial — Part II: Extend PyPOTS to Specialties

## Comprehensive Developer Guide: Architecture, Custom Models across Tasks, Multi-Optimizer GANs, Non-NN Algorithms, Testing, & Open-Source Contribution

> **Welcome to Part II of the PyPOTS KDD 2026 Hands-on Tutorial!**
>
> Part II is specially crafted for **developers, researchers, and core contributors** who want to extend PyPOTS for novel algorithms, domain-specific constraints, and research benchmarks. Based directly on the official PyPOTS Developer Documentation (`docs/dev_*.rst`), this notebook provides a comprehensive, hands-on journey from internal architectural contracts to contribution-ready PRs.
>
> ### 🗺️ Roadmap & Curriculum Outline:
> - **0. Environment & Dependency Setup**: Imports, version checks, global seeds.
> - **II.1 PyPOTS Architecture & Core Contracts**: The 3-Layer structure, 3 Integration Paths, 6 Task Base Contracts, and Data Assembly Hooks (`dev_architecture`, `dev_codebase_map`, `dev_base_classes`, `dev_data_flow`).
> - **II.2 Hands-on Path 1: Standard Neural Network Integration Across Tasks**:
>   - **II.2.A Imputation Task**: `CustomTemporalConvImputer` (`BaseNNImputer` -> `"imputation"`).
>   - **II.2.B Forecasting Task**: `CustomTemporalForecaster` (`BaseNNForecaster` -> `"forecasting"`).
>   - **II.2.C Classification Task**: `CustomTemporalClassifier` (`BaseNNClassifier` -> `"classification_proba"`).
> - **II.3 Hands-on Path 2: Complex NN Integration (GANs & Multi-Optimizer)**: Overriding `_train_model()`, dual optimizers (`G_optimizer`, `D_optimizer`), and checkpoint state preservation (`dev_complex_nn`).
> - **II.4 Hands-on Path 3: Non-NN Algorithm Integration**: Inheriting `BaseImputer` directly for rule-based or statistical algorithms without neural loops (`dev_non_nn`).
> - **II.5 Supporting Domain Constraints & Custom Objectives**: Irregular time sampling (`deltas`), feature-wise masking, and custom `Criterion` subclasses (`WeightedMSE`).
> - **II.6 Benchmark Integration, Testing & Common Integration Mistakes**: Standardized unit testing (`test_0_fit` -> `test_4_lazy_loading`), pytest execution, HDF5 lazy loading, and triage checklist (`dev_testing`, `dev_common_mistakes`).
> - **II.7 Open-Source Workflow & Contribution Practice**: Code formatting (Black), NumPy docstring standard, `pypots-cli dev`, and PR submission checklist (`dev_quality`, `dev_integration_guide`).
> - **II.8 Open-Ended Hands-On Assignment**: Adapting PyPOTS to your own real-world dataset (reusing `benchpots.utils.sliding_window` to slice continuous 2D time series into 3D samples, missingness simulation, training & evaluation).


---

## 0. Environment & Dependency Setup

Let's check our Python environment, import PyTorch, PyPOTS, PyGrinder, BenchPOTS, and setup reusable utility functions for dataset generation and HDF5 saving.


In [ ]:
import math
import os
import sys
import warnings
from copy import deepcopy
from typing import Dict, Optional, Union, Tuple, List

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# PyPOTS ecosystem core imports
import pypots
from pypots.base import BaseModel, BaseNNModel
from pypots.imputation import SAITS, LOCF
from pypots.imputation.base import BaseImputer, BaseNNImputer
from pypots.forecasting.base import BaseNNForecaster
from pypots.classification.base import BaseNNClassifier
from pypots.imputation.saits.data import DatasetForSAITS
from pypots.data.dataset import BaseDataset
from pypots.data.checking import key_in_data_set
from pypots.data.saving import save_dict_into_h5, load_dict_from_h5
from pypots.nn.modules import ModelCore
from pypots.nn.modules.loss import Criterion, MAE, MSE, CrossEntropy
from pypots.nn.functional import calc_mse, calc_mae
from pypots.optim import Adam
from pypots.optim.base import Optimizer
from pypots.utils.logging import logger
from pygrinder import mcar
from benchpots.utils import sliding_window

print(f"✅ PyPOTS version: {pypots.__version__}")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ System Platform: {sys.platform}")


---

## II.1 PyPOTS Architecture & Core Design Contracts

Before writing code for PyPOTS, developers must understand the framework contracts — **where code belongs, which base class to inherit, and how data flows across boundaries**.

### 1. The Three-Layer Model Architecture

Every neural network model in PyPOTS strictly follows a 3-layer modular hierarchy:

```text
pypots/nn/modules/{model}/       # Layer 1: Pure PyTorch nn.Module (backbone, embeddings, layers)
pypots/{task}/{model}/core.py    # Layer 2: _ModelName(ModelCore) — forward computation & loss/metric logic
pypots/{task}/{model}/model.py   # Layer 3: ModelName(BaseNN{Task}) — public API (fit, predict, optimizers)
pypots/{task}/{model}/data.py    # Custom Dataset class (only when BaseDataset is insufficient)
```

| Layer | File / Location | Responsibility | Rule / Boundary |
| :--- | :--- | :--- | :--- |
| **Layer 1: Backbone** | `pypots/nn/modules/{model}/` | Pure PyTorch `nn.Module` (embeddings, temporal blocks, attention). | **ZERO PyPOTS-specific imports** (only `pypots.nn.modules.*`). Must be reusable across tasks. |
| **Layer 2: Core** | `pypots/{task}/{model}/core.py` | Inherits `ModelCore`. Computes forward pass, builds result dict, calculates training loss (`"loss"`) and validation metric (`"metric"`) when `calc_criterion=True`. | Receives input dict from wrapper. Focuses strictly on model computation without wrapper orchestration. |
| **Layer 3: Wrapper API** | `pypots/{task}/{model}/model.py` | Inherits `BaseNN{Task}`. Exposes public API (`fit()`, `predict()`, `impute()`), manages dataloaders, initializes optimizers (`self.optimizer.init_optimizer`), transfers tensors to device, and handles training loops. | User-facing contract. Orchestrates input assembly for train, val, and test stages. |
| **Dataset** | `pypots/{task}/{model}/data.py` | Inherits `BaseDataset`. Normalizes array/file inputs, creates missing masks or artificial missingness. | **Only add** when `BaseDataset` cannot express your model's sample contract. |

### 2. Codebase Map & Module Boundaries

- `pypots/base.py`: Framework contracts (`BaseModel`, `BaseNNModel`).
- `pypots/<task>/`: Task packages (`imputation`, `forecasting`, `classification`, `clustering`, `anomaly_detection`, `representation`). Each task contains `base.py`, `template/`, and model folders.
- `pypots/data/`: `dataset/base.py` (`BaseDataset`), `checking.py` (`key_in_data_set`), `saving/` (`save_dict_into_h5`), `utils.py`.
- `pypots/nn/`: Reusable PyTorch modules (`ModelCore`, `Criterion`, `MAE`, `MSE`, `functional/`).
- `pypots/optim/`: `Optimizer` base class and concrete wrappers (`Adam`, `SGD`, `AdamW`).

### 3. The Three Integration Paths Decision Matrix

Before touching implementation code, decide which path your model belongs to:

| Path | When to Use | Reference Model | Base Classes |
| :--- | :--- | :--- | :--- |
| **Standard NN** | One optimizer, default training loop. Most deep learning models fall here. | `SAITS` (`pypots/imputation/saits/`) | `BaseNN{Task}` + `ModelCore` |
| **Complex NN** | Multiple optimizers (e.g. GAN G & D), alternating update schedules, pretraining stages. | `USGAN` (`pypots/imputation/usgan/`) | `BaseNN{Task}` + `ModelCore` (override `_train_model()`) |
| **Non-NN** | Rule-based, statistical, or algorithmic models. No gradients, no optimizer. | `LOCF` (`pypots/imputation/locf/`) | `Base{Task}` directly (e.g. `BaseImputer`) |

### 4. Six Supported Task Contracts & Result Keys

Each task enforces a public result key returned by `predict()`:

| Task | NN Base Class | Non-NN Base Class | Required Result Key | Public Helper Method |
| :--- | :--- | :--- | :--- | :--- |
| Imputation | `BaseNNImputer` | `BaseImputer` | `"imputation"` | `impute()` |
| Forecasting | `BaseNNForecaster` | `BaseForecaster` | `"forecasting"` | `forecast()` |
| Classification | `BaseNNClassifier` | `BaseClassifier` | `"classification_proba"` | `classify()` |
| Anomaly Detection | `BaseNNDetector` | `BaseDetector` | `"anomaly_detection"` | `detect()` |
| Clustering | `BaseNNClusterer` | `BaseClusterer` | `"clustering"` | `cluster()` |
| Representation | `BaseNNRepresentor` | `BaseRepresentor` | `"representation"` | `represent()` |


---

## II.2 Hands-on Path 1: Standard Neural Network Integration Across Tasks

PyPOTS's 3-Layer architecture is designed to be **task-agnostic**. The Layer 1 backbone can be shared, while Layer 2 Core and Layer 3 Wrapper adapt to specific tasks.

Below we present complete, runnable implementations across 3 key tasks:
1. **II.2.A Imputation Task**: Imputing missing values in time series (`CustomTemporalConvImputer`).
2. **II.2.B Forecasting Task**: Predicting future time steps (`CustomTemporalForecaster`).
3. **II.2.C Classification Task**: Classifying multivariate time series sequences (`CustomTemporalClassifier`).


### II.2.A Imputation Task Example (`CustomTemporalConvImputer`)

Inherits `BaseNNImputer`. Must return result key `"imputation"` from `predict()`.

In [ ]:
class CustomTemporalBlock(nn.Module):
    """Layer 1 Backbone: 1D Temporal Convolutional Block with Residual Connection.
    Pure PyTorch module — no PyPOTS wrapper dependencies.
    """
    def __init__(self, n_features: int, d_model: int, kernel_size: int = 3):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(n_features, d_model, kernel_size=kernel_size, padding=padding)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(d_model, n_features, kernel_size=kernel_size, padding=padding)
        self.norm = nn.LayerNorm(n_features)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = x
        x_trans = x.transpose(1, 2)
        h = self.conv1(x_trans)
        h = self.relu(h)
        out = self.conv2(h).transpose(1, 2)
        return self.norm(res + out)


In [ ]:
class _CustomTemporalConvImputer(ModelCore):
    """Layer 2 Core: Computation & loss handling for CustomTemporalConvImputer."""
    def __init__(self, n_steps: int, n_features: int, d_model: int, training_loss: Criterion, validation_metric: Criterion):
        super().__init__()
        self.n_steps, self.n_features = n_steps, n_features
        self.training_loss = training_loss
        self.validation_metric = training_loss if validation_metric.__class__.__name__ == "Criterion" else validation_metric
        self.backbone = CustomTemporalBlock(n_features, d_model)
        self.output_projection = nn.Linear(n_features, n_features)

    def forward(self, inputs: dict, calc_criterion: bool = False) -> dict:
        X, missing_mask = inputs["X"], inputs["missing_mask"]
        X_filled = torch.nan_to_num(X, nan=0.0)
        repr_h = self.backbone(X_filled)
        reconstruction = self.output_projection(repr_h)
        imputed_data = X_filled * missing_mask + reconstruction * (1 - missing_mask)
        results = {"imputation": imputed_data}
        if calc_criterion:
            X_ori, indicating_mask = inputs["X_ori"], inputs["indicating_mask"]
            results["loss" if self.training else "metric"] = self.training_loss(reconstruction, X_ori, indicating_mask) if self.training else self.validation_metric(reconstruction, X_ori, indicating_mask)
        return results


In [ ]:
class CustomTemporalConvImputer(BaseNNImputer):
    """Layer 3 Wrapper: Public API for CustomTemporalConvImputer."""
    def __init__(
        self, n_steps: int, n_features: int, d_model: int = 32, batch_size: int = 32,
        epochs: int = 10, patience: Optional[int] = None, training_loss: Union[Criterion, type] = MAE,
        validation_metric: Union[Criterion, type] = MSE, optimizer: Union[Optimizer, type] = Adam,
        num_workers: int = 0, device: Optional[Union[str, torch.device, list]] = None, saving_path: Optional[str] = None,
        model_saving_strategy: Optional[str] = "best", verbose: bool = True,
    ):
        super().__init__(training_loss=training_loss, validation_metric=validation_metric, batch_size=batch_size, epochs=epochs, patience=patience, num_workers=num_workers, device=device, saving_path=saving_path, model_saving_strategy=model_saving_strategy, verbose=verbose)
        self.n_steps, self.n_features, self.d_model = n_steps, n_features, d_model
        self.model = _CustomTemporalConvImputer(n_steps=n_steps, n_features=n_features, d_model=d_model, training_loss=self.training_loss, validation_metric=self.validation_metric)
        self._print_model_size()
        self._send_model_to_given_device()
        self.optimizer = optimizer if isinstance(optimizer, Optimizer) else optimizer()
        self.optimizer.init_optimizer(self.model.parameters())
        
    def _assemble_input_for_training(self, data: list) -> dict:
        indices, X, missing_mask, X_ori, indicating_mask = self._send_data_to_given_device(data)
        return {"X": X, "missing_mask": missing_mask, "X_ori": X_ori, "indicating_mask": indicating_mask}
        
    def _assemble_input_for_validating(self, data: list) -> dict:
        return self._assemble_input_for_training(data)
        
    def _assemble_input_for_testing(self, data: list) -> dict:
        indices, X, missing_mask = self._send_data_to_given_device(data)
        return {"X": X, "missing_mask": missing_mask}

    def fit(self, train_set: Union[dict, str], val_set: Optional[Union[dict, str]] = None, file_type: str = "hdf5") -> None:
        train_dataset = DatasetForSAITS(train_set, return_X_ori=False, return_y=False, file_type=file_type)
        train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_dataloader = DataLoader(DatasetForSAITS(val_set, return_X_ori=True, return_y=False, file_type=file_type), batch_size=self.batch_size, shuffle=False) if val_set is not None else None
        self._train_model(train_dataloader, val_dataloader)
        self.model.load_state_dict(self.best_model_dict)


### II.2.B Forecasting Task Example (`CustomTemporalForecaster`)

Inherits `BaseNNForecaster`. Predicts `n_pred_steps` into the future. Must return result key `"forecasting"` from `predict()`.

In [ ]:
class _CustomTemporalForecaster(ModelCore):
    """Layer 2 Core for Forecasting Task."""
    def __init__(self, n_steps: int, n_features: int, n_pred_steps: int, d_model: int, training_loss: Criterion, validation_metric: Criterion):
        super().__init__()
        self.n_steps, self.n_features, self.n_pred_steps = n_steps, n_features, n_pred_steps
        self.training_loss = training_loss
        self.validation_metric = training_loss if validation_metric.__class__.__name__ == "Criterion" else validation_metric
        self.backbone = CustomTemporalBlock(n_features, d_model)
        self.forecasting_head = nn.Linear(n_steps * n_features, n_pred_steps * n_features)

    def forward(self, inputs: dict, calc_criterion: bool = False) -> dict:
        X = inputs["X"]
        X_filled = torch.nan_to_num(X, nan=0.0)
        h = self.backbone(X_filled)
        flat_h = h.reshape(h.size(0), -1)
        pred = self.forecasting_head(flat_h).reshape(h.size(0), self.n_pred_steps, self.n_features)
        results = {"forecasting": pred}
        if calc_criterion:
            X_pred = inputs["X_pred"]
            target_mask = (~torch.isnan(X_pred)).float()
            X_pred_clean = torch.nan_to_num(X_pred, nan=0.0)
            results["loss" if self.training else "metric"] = self.training_loss(pred, X_pred_clean, target_mask) if self.training else self.validation_metric(pred, X_pred_clean, target_mask)
        return results

class CustomTemporalForecaster(BaseNNForecaster):
    """Layer 3 Wrapper API for Forecasting Task."""
    def __init__(
        self, n_steps: int, n_features: int, n_pred_steps: int, d_model: int = 32, batch_size: int = 32,
        epochs: int = 5, training_loss: Union[Criterion, type] = MAE, validation_metric: Union[Criterion, type] = MSE,
        optimizer: Union[Optimizer, type] = Adam, device: Optional[Union[str, torch.device, list]] = None, verbose: bool = True
    ):
        super().__init__(training_loss=training_loss, validation_metric=validation_metric, batch_size=batch_size, epochs=epochs, device=device, verbose=verbose)
        self.n_steps, self.n_features, self.n_pred_steps = n_steps, n_features, n_pred_steps
        self.model = _CustomTemporalForecaster(n_steps, n_features, n_pred_steps, d_model, self.training_loss, self.validation_metric)
        self._send_model_to_given_device()
        self.optimizer = optimizer if isinstance(optimizer, Optimizer) else optimizer()
        self.optimizer.init_optimizer(self.model.parameters())

    def _assemble_input_for_training(self, data: list) -> dict:
        indices, X, missing_mask, X_pred, X_pred_missing_mask = self._send_data_to_given_device(data)
        return {"X": X, "missing_mask": missing_mask, "X_pred": X_pred}

    def _assemble_input_for_validating(self, data: list) -> dict:
        return self._assemble_input_for_training(data)

    def _assemble_input_for_testing(self, data: list) -> dict:
        indices, X, missing_mask = self._send_data_to_given_device(data)
        return {"X": X, "missing_mask": missing_mask}

    def fit(self, train_set: Union[dict, str], val_set: Optional[Union[dict, str]] = None, file_type: str = "hdf5") -> None:
        train_dataset = BaseDataset(train_set, return_X_ori=False, return_X_pred=True, return_y=False, file_type=file_type)
        train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_dataloader = DataLoader(BaseDataset(val_set, return_X_ori=False, return_X_pred=True, return_y=False, file_type=file_type), batch_size=self.batch_size, shuffle=False) if val_set is not None else None
        self._train_model(train_dataloader, val_dataloader)
        if hasattr(self, "best_model_dict") and self.best_model_dict is not None:
            self.model.load_state_dict(self.best_model_dict)


### II.2.C Classification Task Example (`CustomTemporalClassifier`)

Inherits `BaseNNClassifier`. Classifies time-series samples into `n_classes`. Must return result key `"classification_proba"` from Core `forward()`.

In [ ]:
class _CustomTemporalClassifier(ModelCore):
    """Layer 2 Core for Classification Task."""
    def __init__(self, n_steps: int, n_features: int, n_classes: int, d_model: int):
        super().__init__()
        self.backbone = CustomTemporalBlock(n_features, d_model)
        self.classifier_head = nn.Linear(n_steps * n_features, n_classes)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, inputs: dict, calc_criterion: bool = False) -> dict:
        X = inputs["X"]
        X_filled = torch.nan_to_num(X, nan=0.0)
        h = self.backbone(X_filled)
        logits = self.classifier_head(h.reshape(h.size(0), -1))
        proba = torch.softmax(logits, dim=-1)
        results = {"classification_proba": proba}
        if calc_criterion:
            y = inputs["y"].long()
            loss = self.loss_fn(logits, y)
            results["loss" if self.training else "metric"] = loss
        return results

class CustomTemporalClassifier(BaseNNClassifier):
    """Layer 3 Wrapper API for Classification Task."""
    def __init__(
        self, n_steps: int, n_features: int, n_classes: int, d_model: int = 32, batch_size: int = 32,
        epochs: int = 5, optimizer: Union[Optimizer, type] = Adam, device: Optional[Union[str, torch.device, list]] = None, verbose: bool = True
    ):
        super().__init__(n_classes=n_classes, training_loss=CrossEntropy(), validation_metric=CrossEntropy(), batch_size=batch_size, epochs=epochs, device=device, verbose=verbose)
        self.n_steps, self.n_features, self.n_classes = n_steps, n_features, n_classes
        self.model = _CustomTemporalClassifier(n_steps, n_features, n_classes, d_model)
        self._send_model_to_given_device()
        self.optimizer = optimizer if isinstance(optimizer, Optimizer) else optimizer()
        self.optimizer.init_optimizer(self.model.parameters())

    def _assemble_input_for_training(self, data: list) -> dict:
        indices, X, missing_mask, y = self._send_data_to_given_device(data)
        return {"X": X, "missing_mask": missing_mask, "y": y}

    def _assemble_input_for_validating(self, data: list) -> dict:
        return self._assemble_input_for_training(data)

    def _assemble_input_for_testing(self, data: list) -> dict:
        indices, X, missing_mask = self._send_data_to_given_device(data)
        return {"X": X, "missing_mask": missing_mask}

    def fit(self, train_set: Union[dict, str], val_set: Optional[Union[dict, str]] = None, file_type: str = "hdf5") -> None:
        train_dataset = BaseDataset(train_set, return_X_ori=False, return_X_pred=False, return_y=True, file_type=file_type)
        train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_dataloader = DataLoader(BaseDataset(val_set, return_X_ori=False, return_X_pred=False, return_y=True, file_type=file_type), batch_size=self.batch_size, shuffle=False) if val_set is not None else None
        self._train_model(train_dataloader, val_dataloader)
        if hasattr(self, "best_model_dict") and self.best_model_dict is not None:
            self.model.load_state_dict(self.best_model_dict)


### Fit & Evaluate All 3 Task Models

Let's test Imputation, Forecasting, and Classification models on synthetic POTS data.


In [ ]:
# Generate synthetic dataset for all 3 tasks
np.random.seed(42)
n_samples, n_steps, n_features, n_pred_steps, n_classes = 100, 24, 5, 6, 2
raw_X = np.random.randn(n_samples, n_steps, n_features).astype(np.float32)
mask = (np.random.rand(n_samples, n_steps, n_features) > 0.2).astype(np.float32)
X_with_nan = raw_X.copy()
X_with_nan[mask == 0] = np.nan

# 1. Imputation data
imp_train = {"X": X_with_nan[:80]}
imp_val = {"X": X_with_nan[80:], "X_ori": raw_X[80:]}
imp_test = {"X": X_with_nan[80:]}

# 2. Forecasting data
raw_X_pred = np.random.randn(n_samples, n_pred_steps, n_features).astype(np.float32)
fore_train = {"X": X_with_nan[:80], "X_pred": raw_X_pred[:80]}
fore_val = {"X": X_with_nan[80:], "X_pred": raw_X_pred[80:]}
fore_test = {"X": X_with_nan[80:]}

# 3. Classification data
y_labels = np.random.randint(0, n_classes, size=n_samples)
cls_train = {"X": X_with_nan[:80], "y": y_labels[:80]}
cls_val = {"X": X_with_nan[80:], "y": y_labels[80:]}
cls_test = {"X": X_with_nan[80:]}

# Execute Imputer
print("--- 1. Testing Imputation Task ---")
imputer = CustomTemporalConvImputer(n_steps, n_features, d_model=16, epochs=2)
imputer.fit(imp_train, imp_val)
imp_res = imputer.predict(imp_test)
print(f"Imputation output shape: {imp_res['imputation'].shape}")

# Execute Forecaster
print("\n--- 2. Testing Forecasting Task ---")
forecaster = CustomTemporalForecaster(n_steps, n_features, n_pred_steps, d_model=16, epochs=2)
forecaster.fit(fore_train, fore_val)
fore_res = forecaster.predict(fore_test)
print(f"Forecasting output shape: {fore_res['forecasting'].shape}")

# Execute Classifier
print("\n--- 3. Testing Classification Task ---")
classifier = CustomTemporalClassifier(n_steps, n_features, n_classes, d_model=16, epochs=2)
classifier.fit(cls_train, cls_val)
cls_res = classifier.predict(cls_test)
print(f"Classification proba output shape: {cls_res['classification_proba'].shape}")


---

## II.3 Hands-on Path 2: Implementing a Complex NN Model (GANs & Multi-Optimizer)

When a model requires **multiple optimizers** (e.g. Generator + Discriminator), alternating update schedules, or pretraining stages, the default `_train_model()` in `BaseNNModel` is no longer sufficient (`dev_complex_nn.rst`).

### Key Invariants to Preserve When Overriding `_train_model()`:
1. **Reset best-loss state** at the start (`self.best_loss = float("inf")`).
2. **Run explicit train and validation phases**.
3. **Track best model dictionary** (`self.best_model_dict = deepcopy(self.model.state_dict())`).
4. **Update patience counter** for early stopping.
5. **Restore best checkpoint** after training loop completes (`self.model.load_state_dict(self.best_model_dict)`).

Let's build a runnable **`CustomGANImputer`** demonstrating multi-optimizer training!


In [ ]:
class _CustomGANCore(ModelCore):
    """Layer 2 Core for GAN Imputer handling generator and discriminator passes."""
    def __init__(self, n_steps: int, n_features: int, d_model: int, validation_metric: Criterion):
        super().__init__()
        self.validation_metric = validation_metric
        self.generator = nn.Sequential(nn.Linear(n_features, d_model), nn.ReLU(), nn.Linear(d_model, n_features))
        self.discriminator = nn.Sequential(nn.Linear(n_features, d_model), nn.ReLU(), nn.Linear(d_model, n_features), nn.Sigmoid())

    def forward(self, inputs: dict, training_object: str = "generator", calc_criterion: bool = False) -> dict:
        X, missing_mask = inputs["X"], inputs["missing_mask"]
        X_filled = torch.nan_to_num(X, nan=0.0)
        g_output = self.generator(X_filled)
        imputed_data = X_filled * missing_mask + g_output * (1 - missing_mask)
        results = {"imputation": imputed_data}
        if training_object == "discriminator":
            d_pred = self.discriminator(imputed_data.detach())
            results["D_loss"] = torch.mean((d_pred - missing_mask) ** 2)
        elif training_object == "generator":
            d_pred = self.discriminator(imputed_data)
            results["G_loss"] = torch.mean((d_pred - 1.0) ** 2) + torch.mean((g_output - X_filled) ** 2)
        if calc_criterion:
            results["metric"] = self.validation_metric(imputed_data, inputs["X_ori"], inputs["indicating_mask"])
        return results


In [ ]:
class CustomGANImputer(BaseNNImputer):
    """Layer 3 Wrapper for CustomGANImputer demonstrating dual-optimizer training."""
    def __init__(
        self, n_steps: int, n_features: int, d_model: int = 16, batch_size: int = 16,
        epochs: int = 5, patience: Optional[int] = None, G_optimizer: Union[Optimizer, type] = Adam,
        D_optimizer: Union[Optimizer, type] = Adam, validation_metric: Union[Criterion, type] = MSE,
        device: Optional[Union[str, torch.device, list]] = None, verbose: bool = True,
    ):
        super().__init__(training_loss=MAE(), validation_metric=validation_metric, batch_size=batch_size, epochs=epochs, patience=patience, device=device, verbose=verbose)
        self.n_steps, self.n_features = n_steps, n_features
        self.model = _CustomGANCore(n_steps, n_features, d_model, self.validation_metric)
        self._send_model_to_given_device()
        self.G_optimizer = G_optimizer() if not isinstance(G_optimizer, Optimizer) else G_optimizer
        self.G_optimizer.init_optimizer(self.model.generator.parameters())
        self.D_optimizer = D_optimizer() if not isinstance(D_optimizer, Optimizer) else D_optimizer
        self.D_optimizer.init_optimizer(self.model.discriminator.parameters())

    def _assemble_input_for_training(self, data: list) -> dict:
        indices, X, missing_mask, X_ori, indicating_mask = self._send_data_to_given_device(data)
        return {"X": X, "missing_mask": missing_mask, "X_ori": X_ori, "indicating_mask": indicating_mask}

    def _assemble_input_for_validating(self, data: list) -> dict:
        return self._assemble_input_for_training(data)

    def _assemble_input_for_testing(self, data: list) -> dict:
        indices, X, missing_mask = self._send_data_to_given_device(data)
        return {"X": X, "missing_mask": missing_mask}

    def _train_model(self, train_dataloader: DataLoader, val_dataloader: Optional[DataLoader] = None) -> None:
        self.best_loss, self.best_epoch, patience_counter = float("inf"), 0, 0
        for epoch in range(self.epochs):
            self.model.train()
            g_loss_collector, d_loss_collector = [], []
            for idx, data in enumerate(train_dataloader):
                inputs = self._assemble_input_for_training(data)
                self.D_optimizer.zero_grad()
                d_loss = self.model(inputs, training_object="discriminator")["D_loss"]
                d_loss.backward()
                self.D_optimizer.step()
                d_loss_collector.append(d_loss.item())
                self.G_optimizer.zero_grad()
                g_loss = self.model(inputs, training_object="generator")["G_loss"]
                g_loss.backward()
                self.G_optimizer.step()
                g_loss_collector.append(g_loss.item())
            if val_dataloader is not None:
                self.model.eval()
                val_metrics = []
                with torch.no_grad():
                    for idx, data in enumerate(val_dataloader):
                        val_metrics.append(self.model(self._assemble_input_for_validating(data), calc_criterion=True)["metric"].item())
                mean_val_loss = float(np.mean(val_metrics))
                if mean_val_loss < self.best_loss:
                    self.best_loss, self.best_epoch = mean_val_loss, epoch
                    self.best_model_dict = deepcopy(self.model.state_dict())
                    patience_counter = 0
                else:
                    patience_counter += 1
                if self.patience is not None and patience_counter >= self.patience:
                    break

    def fit(self, train_set: Union[dict, str], val_set: Optional[Union[dict, str]] = None, file_type: str = "hdf5") -> None:
        train_dataloader = DataLoader(DatasetForSAITS(train_set, return_X_ori=False, return_y=False, file_type=file_type), batch_size=self.batch_size, shuffle=True)
        val_dataloader = DataLoader(DatasetForSAITS(val_set, return_X_ori=True, return_y=False, file_type=file_type), batch_size=self.batch_size, shuffle=False) if val_set is not None else None
        self._train_model(train_dataloader, val_dataloader)
        if hasattr(self, "best_model_dict") and self.best_model_dict is not None:
            self.model.load_state_dict(self.best_model_dict)


In [ ]:
# Test CustomGANImputer end-to-end
gan_imputer = CustomGANImputer(n_steps=n_steps, n_features=n_features, d_model=16, epochs=2, batch_size=16)
print("--- Training CustomGANImputer (Complex NN / Multi-Optimizer Path) ---")
gan_imputer.fit(train_set=imp_train, val_set=imp_val)
gan_results = gan_imputer.predict(imp_test)
print(f"GAN Imputed output shape: {gan_results['imputation'].shape}")


---

## II.4 Hands-on Path 3: Implementing a Non-NN Algorithm

For statistical, rule-based, or heuristic algorithms (such as `LOCF`, `Mean`, `Median`, `Lerp`, matrix factorization `TRMF`), inheriting `BaseNNModel` would introduce fake optimizers and useless hooks (`dev_non_nn.rst`).

Instead, inherit **`BaseImputer`** directly (or `BaseForecaster`, `BaseClassifier`):
- `fit()`: Explicit no-op with a warning (or learns non-neural parameters if stateful).
- `predict()`: Handles both dictionary input and HDF5 file path input directly.
- `impute()`: Public helper returning the imputed array.


In [ ]:
class CustomInterpolationImputer(BaseImputer):
    """Non-NN Path: Linear interpolation imputer inheriting BaseImputer directly."""
    def __init__(self, device: Optional[Union[str, torch.device, list]] = None):
        super().__init__(device=device)

    def fit(self, train_set: Union[dict, str], val_set: Optional[Union[dict, str]] = None, file_type: str = "hdf5") -> None:
        warnings.warn("CustomInterpolationImputer is a non-NN statistical model without trainable parameters. Call predict() directly.")

    def predict(self, test_set: Union[dict, str], file_type: str = "hdf5", **kwargs) -> dict:
        if isinstance(test_set, str):
            with h5py.File(test_set, "r") as f:
                X = f["X"][:]
        else:
            X = test_set["X"]
        assert len(X.shape) == 3, f"Input X must have shape [n_samples, n_steps, n_features], got {X.shape}"
        imputed_X = X.copy()
        n_samples, n_steps, n_features = imputed_X.shape
        for i in range(n_samples):
            for j in range(n_features):
                series = imputed_X[i, :, j]
                nans = np.isnan(series)
                if nans.all(): series[:] = 0.0
                elif nans.any():
                    x_indices = np.arange(n_steps)
                    series[nans] = np.interp(x_indices[nans], x_indices[~nans], series[~nans])
        return {"imputation": imputed_X}


In [ ]:
# Test CustomInterpolationImputer end-to-end
non_nn_imputer = CustomInterpolationImputer()
non_nn_imputer.fit(imp_train)
non_nn_results = non_nn_imputer.predict(imp_test)
print(f"Non-NN Imputed shape: {non_nn_results['imputation'].shape}")


---

## II.5 Supporting Domain Constraints & Custom Objectives

Real-world time series (healthcare, IoT, finance) often bring specific domain constraints:

1. **Irregular Time Intervals (`deltas`)**: Pass relative time steps $D_{t,i} = t_i - t_{i-1}$ as input tensors into Layer 2 core.
2. **Channel/Feature Masking**: Apply feature-wise weight matrices to penalize specific sensors or vital signs.
3. **Custom Loss Objectives**: Subclass `Criterion` to build domain-specific loss functions.


In [ ]:
class WeightedMSE(Criterion):
    """Custom domain-constrained loss: Feature-Weighted Mean Squared Error."""
    def __init__(self, feature_weights: Optional[torch.Tensor] = None):
        super().__init__()
        self.feature_weights = feature_weights

    def forward(self, predictions: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        diff = (predictions - targets) ** 2
        if self.feature_weights is not None:
            diff = diff * self.feature_weights.to(predictions.device)
        return (diff * mask).sum() / (mask.sum() + 1e-5)

weights = torch.tensor([3.0, 1.0, 1.0, 1.0, 5.0])
custom_loss = WeightedMSE(feature_weights=weights)
pred, target, mask_tensor = torch.randn(10, 24, 5), torch.randn(10, 24, 5), torch.ones(10, 24, 5)
print(f"Calculated feature-weighted loss: {custom_loss(pred, target, mask_tensor).item():.4f}")


---

## II.6 Benchmark Integration, Testing & Common Integration Mistakes

PyPOTS enforces strict unit testing standards to ensure research reproducibility and code reliability (`dev_testing.rst`, `dev_common_mistakes.rst`).

### 1. Standardized PyPOTS Unit Test Structure

Every model test file in `tests/<task>/<model>.py` defines 5 standardized test methods:

```python
# tests/imputation/test_custom_model.py
import unittest
import numpy as np
import pytest
from pypots.imputation import CustomTemporalConvImputer
from tests.global_test_config import DATA, EPOCHS, DEVICE, TRAIN_SET, VAL_SET, TEST_SET, GENERAL_H5_TRAIN_SET_PATH, GENERAL_H5_VAL_SET_PATH, GENERAL_H5_TEST_SET_PATH

class TestCustomModel(unittest.TestCase):
    model = CustomTemporalConvImputer(n_steps=DATA["n_steps"], n_features=DATA["n_features"], epochs=EPOCHS, device=DEVICE)

    @pytest.mark.xdist_group(name="imputation-custom_model")
    def test_0_fit(self):
        self.model.fit(TRAIN_SET, VAL_SET)

    @pytest.mark.xdist_group(name="imputation-custom_model")
    def test_1_impute(self):
        results = self.model.predict(TEST_SET)
        assert "imputation" in results and not np.isnan(results["imputation"]).any()

    @pytest.mark.xdist_group(name="imputation-custom_model")
    def test_2_parameters(self):
        assert hasattr(self.model, "best_loss") and self.model.best_model_dict is not None

    @pytest.mark.xdist_group(name="imputation-custom_model")
    def test_3_saving_path(self):
        self.model.save("checkpoint.pypots")
        self.model.load("checkpoint.pypots")

    @pytest.mark.xdist_group(name="imputation-custom_model")
    def test_4_lazy_loading(self):
        self.model.fit(GENERAL_H5_TRAIN_SET_PATH, GENERAL_H5_VAL_SET_PATH)
        results = self.model.predict(GENERAL_H5_TEST_SET_PATH)
        assert not np.isnan(results["imputation"]).any()
```

### 2. Testing Lazy Loading (HDF5 File Input)

Let's test saving our synthetic data into HDF5 files using `save_dict_into_h5` and verifying that our custom model fits and predicts directly from file paths!


In [ ]:
# Save train_set and val_set to HDF5 files for lazy-loading test
h5_train_path, h5_val_path, h5_test_path = "tmp_train_set.h5", "tmp_val_set.h5", "tmp_test_set.h5"
save_dict_into_h5(imp_train, h5_train_path)
save_dict_into_h5(imp_val, h5_val_path)
save_dict_into_h5(imp_test, h5_test_path)

print("--- Testing HDF5 Lazy-Loading Input ---")
h5_imputer = CustomTemporalConvImputer(n_steps=n_steps, n_features=n_features, d_model=16, epochs=2)
h5_imputer.fit(h5_train_path, h5_val_path)
h5_results = h5_imputer.predict(h5_test_path)
print(f"HDF5 Lazy-loading imputed shape: {h5_results['imputation'].shape}")

for p in [h5_train_path, h5_val_path, h5_test_path]:
    if os.path.exists(p): os.remove(p)


### 3. Top Common Integration Mistakes & Fixes

| Integration Mistake | Symptom | How to Fix |
| :--- | :--- | :--- |
| **1. Trusting Placeholders over Contracts** | Copying template placeholder keys like `"output"`. | Read task base class (e.g. `BaseNNImputer`). Output dict **must** include task key `"imputation"`. |
| **2. Input Key Drift across Stages** | Training works, but validation/testing crashes with `KeyError` or unpacking errors. | `_assemble_input_for_training` and `_validating` can expect `X_ori`, but `_assemble_input_for_testing` **must stay minimal** (`X`, `missing_mask`). |
| **3. Choosing the Wrong Path** | Rule-based model wrapped in fake optimizer, or GAN model forced into standard single loop. | Refer to 3-Path matrix: Standard NN vs Complex NN (custom `_train_model`) vs Non-NN (`BaseImputer`). |
| **4. Breaking Checkpoint Semantics** | Custom loop runs, but best model is never loaded or patience stops working. | Always track `self.best_loss`, save `self.best_model_dict`, update patience, and run `self.model.load_state_dict(self.best_model_dict)`. |
| **5. Unnecessary `data.py` Creation** | Adding `data.py` just to rename dictionary keys. | Reuse `BaseDataset` unless you need artificial masking (like SAITS) or stage-dependent extra tensors. |
| **6. Shipping Without Lazy-Loading Tests** | Dict input works, but HDF5 file path input fails in CI. | Always test both dict input and HDF5 file input before submitting PR. |


---

## II.7 Open-Source Workflow & Contribution Practice

Follow this step-by-step checklist to submit your contribution back to PyPOTS (`dev_quality.rst`, `dev_integration_guide.rst`):

### 1. Code Style & Quality Requirements
- **Code Formatter**: Black with line length **120**.
- **Docstrings**: NumPy-style docstrings for all public classes, methods, and functions.
- **Lint Command**:
  ```bash
  pypots-cli dev --lint_code
  # Or manually:
  flake8 .
  ```

### 2. Local Testing Commands
```bash
# Generate shared test dataset
python tests/global_test_config.py

# Run targeted model unit test
pytest -rA tests/imputation/your_model.py -n 1

# Full regression (when modifying shared modules base.py/data/nn/optim)
pytest -rA -s tests/*/* -n 1 --cov=pypots --dist=loadgroup --cov-config=.coveragerc
```

### 3. Pre-PR Checklist & Evidence Template

```markdown
## PR Description & Testing Evidence

### Changes Proposed
- Implemented [Model Name] under `pypots/imputation/your_model/` following the 3-Layer architecture.
- Exported model in `pypots/imputation/__init__.py`.
- Added unit test in `tests/imputation/your_model.py`.

### Local Validation Evidence
- Environment: Python 3.10, PyTorch 2.1, macOS / Linux
- Commands Run:
  ```bash
  pypots-cli dev --lint_code
  python tests/global_test_config.py
  pytest -rA tests/imputation/your_model.py -n 1
  ```
- Results: All 5 tests passed (fit, predict, parameters, save/load, lazy loading). 0 lint errors.
```


---

## II.8 Open-Ended Hands-On Assignment: Adapting PyPOTS to Your Own Real-World Dataset

> **🎯 Assignment Objective:**
> Apply PyPOTS to a real-world continuous time-series dataset from your own domain (e.g. healthcare EHRs, IoT sensors, weather monitoring, traffic, or financial logs). Learn how to slice long 2D sequence data `[total_steps, n_features]` into a 3D dataset array `[n_samples, n_steps, n_features]` via BenchPOTS's built-in `benchpots.utils.sliding_window` utility, simulate/process missing values using PyGrinder, train a PyPOTS model (e.g., SAITS or LOCF), and evaluate performance.

### 📋 Real-World Preprocessing Pipeline Tasks:
1. **Continuous 2D Time Series -> 3D Sliding Window Slicing (`benchpots.utils.sliding_window`)**:
   - In real-world applications, data usually arrives as continuous long sequences `[total_steps, n_features]`.
   - Directly reuse `benchpots.utils.sliding_window(time_series, window_size, stride)` to produce a 3D NumPy array of shape `[n_samples, window_size, n_features]`.
2. **Missingness Injection & Mask Generation**:
   - Use `pygrinder.mcar(X, p=0.2)` to inject missingness and create ground-truth evaluation pairs (`X` with NaNs and `X_ori`).
3. **Dataset Dictionary Assembly**:
   - Construct `train_set = {"X": train_X_with_nan}`.
   - Construct `val_set = {"X": val_X_with_nan, "X_ori": val_raw_X}`.
   - Construct `test_set = {"X": test_X_with_nan}`.
4. **Train, Predict & Benchmark**:
   - Instantiate PyPOTS model (`SAITS` / `LOCF` / Custom Model).
   - Fit model on `train_set` and `val_set`, generate predictions, and evaluate MSE / MAE metrics using `calc_mse(results["imputation"], test_raw_X, test_indicating_mask)`.

---  
### 🚀 Starter Code Scaffold Reusing `benchpots.utils.sliding_window` (Execute Below):


In [ ]:
# ==========================================================================
# Step 1: Simulate continuous long 2D raw time series [1000 steps, 6 features]
# Replace 'raw_long_2d_series' with your own Pandas DataFrame / NumPy array!
# ==========================================================================
np.random.seed(2026)
total_steps, n_features = 1000, 6
time_axis = np.linspace(0, 20 * np.pi, total_steps)
raw_long_2d_series = np.zeros((total_steps, n_features), dtype=np.float32)
for f in range(n_features):
    raw_long_2d_series[:, f] = np.sin(time_axis + f * 0.5) + 0.1 * np.random.randn(total_steps)

print(f"Raw continuous 2D time series shape: {raw_long_2d_series.shape}")

# Step 2: Slice continuous 2D series into 3D samples using benchpots.utils.sliding_window
window_size = 36  # Each sample will have 36 time steps (n_steps)
stride = 12       # Window step size
custom_raw_3D = sliding_window(raw_long_2d_series, window_size=window_size, stride=stride)
n_samples_custom, n_steps_custom, n_features_custom = custom_raw_3D.shape
print(f"Sliced 3D dataset array shape (via BenchPOTS sliding_window): {custom_raw_3D.shape}")

# Step 3: Missingness simulation with PyGrinder (MCAR 20% rate)
custom_X_masked = mcar(custom_raw_3D, p=0.20)
custom_missing_mask = (~np.isnan(custom_X_masked)).astype(np.float32)
custom_indicating_mask = np.isnan(custom_X_masked).astype(np.float32)

# Train/Val/Test Split (80% train, 10% val, 10% test)
train_idx, val_idx = int(n_samples_custom * 0.8), int(n_samples_custom * 0.9)
my_train_set = {"X": custom_X_masked[:train_idx]}
my_val_set = {"X": custom_X_masked[train_idx:val_idx], "X_ori": custom_raw_3D[train_idx:val_idx]}
my_test_set = {"X": custom_X_masked[val_idx:]}
my_test_raw_X, my_test_indicating_mask = custom_raw_3D[val_idx:], custom_indicating_mask[val_idx:]

# Step 4: Instantiate & Train SAITS Model
print("\n--- 1. Training SAITS Model on BenchPOTS-Sliced Data ---")
saits_model = SAITS(n_steps=n_steps_custom, n_features=n_features_custom, n_layers=2, d_model=32, d_ffn=64, n_heads=4, d_k=8, d_v=8, dropout=0.1, epochs=3, batch_size=16)
saits_model.fit(train_set=my_train_set, val_set=my_val_set)
saits_results = saits_model.predict(my_test_set)
saits_imputed = saits_results["imputation"]
saits_mse = calc_mse(saits_imputed, my_test_raw_X, my_test_indicating_mask)
print(f"SAITS Imputation Performance -> MSE: {saits_mse:.4f}")

# Step 5: Non-NN LOCF Baseline Comparison
print("\n--- 2. Evaluating LOCF Non-NN Baseline ---")
locf_model = LOCF()
locf_results = locf_model.predict(my_test_set)
locf_imputed = locf_results["imputation"]
locf_mse = calc_mse(locf_imputed, my_test_raw_X, my_test_indicating_mask)
print(f"LOCF Imputation Performance -> MSE: {locf_mse:.4f}")

print("\n🎉 Real-World BenchPOTS Sliding Window Assignment Step Completed!")


---
### Summary & Next Steps
🎉 **Congratulations!** You have completed Part II of the PyPOTS KDD 2026 Hands-on Tutorial. You now master PyPOTS's internal architecture, integration paths, multi-task model development (Imputation, Forecasting, Classification), multi-optimizer orchestration, non-NN algorithm wrapping, unit testing, open-source contribution practices, and real-world dataset adaptation. Welcome to the PyPOTS open-source developer community!
